# 📘 Deep Learning Text Generation
## Vanilla RNN vs LSTM vs GRU — Complete Learning Project

---

## 🗺️ Project Workflow Overview

```
Raw Text Corpus
      │
      ▼
  Tokenization  ──→  word → integer mapping
      │
      ▼
N-gram Sequences  ──→  sliding window creates (X, y) pairs
      │
      ▼
  Padding       ──→  all sequences same length
      │
      ▼
One-Hot / Embedding  ──→  dense vector representation
      │
      ├──────────────┬──────────────┐
      ▼              ▼              ▼
  Vanilla RNN      LSTM           GRU
      │              │              │
      └──────────────┴──────────────┘
                     │
                     ▼
          Dense(softmax) → Predict next word
                     │
                     ▼
          Compare Loss / Accuracy / Generated Text
```

---

## 🧠 Theory: Why Sequence Models for Text?

Text is **sequential data** — the meaning of a word depends on what came before it.
Standard feedforward networks have no memory. Recurrent networks solve this by
passing a **hidden state** from one time step to the next.

| Model | Memory | Gates | Use Case |
|-------|--------|-------|----------|
| Vanilla RNN | Short-term only | None | Simple patterns |
| LSTM | Long + Short term | 3 gates | Long sentences, paragraphs |
| GRU | Long + Short term | 2 gates | Faster training, similar results |

---

## ⚙️ Architecture Internals

### Vanilla RNN
```
h_t = tanh(W_h · h_{t-1} + W_x · x_t + b)
```
- Single equation, simple recurrence
- ❌ Suffers **vanishing gradient**: gradients shrink exponentially during backprop
- ❌ Cannot remember information from many steps ago

### LSTM (Long Short-Term Memory)
```
Forget Gate:  f_t = σ(W_f · [h_{t-1}, x_t] + b_f)   ← What to forget?
Input Gate:   i_t = σ(W_i · [h_{t-1}, x_t] + b_i)   ← What new info to add?
Cell Update:  C̃_t = tanh(W_C · [h_{t-1}, x_t] + b_C)
Cell State:   C_t = f_t * C_{t-1} + i_t * C̃_t       ← Memory highway
Output Gate:  o_t = σ(W_o · [h_{t-1}, x_t] + b_o)
Hidden State: h_t = o_t * tanh(C_t)
```
- ✅ Cell state acts as a **conveyor belt** preserving long-range information
- ✅ Gates learn what to remember, forget, and output
- ❌ More parameters → slower training

### GRU (Gated Recurrent Unit)
```
Reset Gate:   r_t = σ(W_r · [h_{t-1}, x_t])    ← How much past to use?
Update Gate:  z_t = σ(W_z · [h_{t-1}, x_t])    ← How much to update?
Candidate:    h̃_t = tanh(W · [r_t * h_{t-1}, x_t])
Hidden State: h_t = (1 - z_t) * h_{t-1} + z_t * h̃_t
```
- ✅ Merges cell state + hidden state → simpler architecture
- ✅ Fewer parameters → **faster** than LSTM
- ✅ Comparable performance on most tasks

## 📦 Step 1: Import Libraries

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Embedding, SimpleRNN, LSTM, GRU, Dense, Dropout, BatchNormalization
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
tf.random.set_seed(42)
np.random.seed(42)

print(f"TensorFlow version : {tf.__version__}")
print(f"GPU available      : {len(tf.config.list_physical_devices('GPU')) > 0}")

## 📥 Step 2: Text Corpus

**Theory — What makes a good corpus?**
- Sufficient repetition so the model learns patterns
- Consistent vocabulary (avoid too many rare words)
- Domain-specific vocabulary for targeted generation

The corpus below covers deep learning terminology — you can swap it
with any text (Shakespeare, song lyrics, your own writing, etc.).

In [ ]:
corpus = '''
deep learning is transforming artificial intelligence and modern technology
recurrent neural networks are designed for sequential data processing
lstm helps remember long term dependencies across many time steps
gru is faster and simpler than lstm yet achieves similar performance
text generation models predict the next word in a sequence
deep learning models can generate meaningful and coherent sentences
neural networks learn patterns from large amounts of training data
attention mechanisms allow models to focus on relevant information
gradient descent optimizes neural network weights during training
embedding layers convert words into dense numerical vector representations
backpropagation through time computes gradients for recurrent networks
deep learning has revolutionized natural language processing and computer vision
transformer models have replaced recurrent networks in many applications
recurrent architectures process sequences step by step maintaining hidden state
the vanishing gradient problem limits vanilla rnn from learning long sequences
'''

lines = [l.strip() for l in corpus.strip().split('\n') if l.strip()]
print(f"Total lines in corpus : {len(lines)}")
print(f"Total characters      : {len(corpus)}")
print(f"\nSample lines:")
for l in lines[:3]:
    print(f"  → {l}")

## 🔤 Step 3: Tokenization & Sequence Preparation

**Theory — N-gram Training Strategy:**

For the sentence `"deep learning is great"`, we create:
```
[deep, learning]         → predict: is
[deep, learning, is]     → predict: great
```
This means every prefix of a sentence becomes a training example.
The model learns to predict the **next word** given **all previous words**.

In [ ]:
# ── Tokenization ──────────────────────────────────────────────────────────────
tokenizer = Tokenizer(oov_token='<OOV>')   # handles unseen words at inference
tokenizer.fit_on_texts(lines)

total_words = len(tokenizer.word_index) + 1
print(f"Vocabulary size : {total_words}")
print(f"Sample mappings : { {k: v for k, v in list(tokenizer.word_index.items())[:8]} }")

# ── Build n-gram sequences ─────────────────────────────────────────────────────
input_sequences = []
for line in lines:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        input_sequences.append(token_list[:i + 1])

max_len = max(len(s) for s in input_sequences)
print(f"\nTotal training sequences : {len(input_sequences)}")
print(f"Max sequence length      : {max_len}")

# ── Pad + split X / y ─────────────────────────────────────────────────────────
padded = np.array(pad_sequences(input_sequences, maxlen=max_len, padding='pre'))

X = padded[:, :-1]          # all tokens except last → features
y = padded[:, -1]           # last token → label (next word index)

print(f"X shape : {X.shape}   (samples × sequence_length)")
print(f"y shape : {y.shape}   (next-word index for each sample)")

# ── Show one example ──────────────────────────────────────────────────────────
idx_to_word = {v: k for k, v in tokenizer.word_index.items()}
sample_x = [idx_to_word.get(i, '') for i in X[5] if i != 0]
sample_y = idx_to_word.get(y[5], '?')
print(f"\nExample → Input : {sample_x}  |  Predict : '{sample_y}'")

## 🏗️ Step 4: Build Models

**Theory — Layer-by-layer explanation:**

```
Input (word indices)  →  shape: (batch, seq_len-1)
        ↓
Embedding             →  shape: (batch, seq_len-1, embed_dim)
   Maps each word index to a dense learnable vector.
   Words with similar meanings cluster together in embedding space.
        ↓
RNN / LSTM / GRU      →  shape: (batch, hidden_units)
   Processes sequence left-to-right, maintaining hidden state.
        ↓
Dropout               →  regularization — randomly zeros activations
        ↓
Dense(softmax)        →  shape: (batch, vocab_size)
   Probability distribution over entire vocabulary.
   argmax → predicted next word index.
```

**Loss: `sparse_categorical_crossentropy`**
- Used when labels are integer indices (not one-hot)
- Measures how surprised the model was by the correct word
- Lower loss = model assigns higher probability to the correct word

In [ ]:
# ── Shared hyperparameters ─────────────────────────────────────────────────────
EMBED_DIM    = 64     # embedding vector size per word
HIDDEN_UNITS = 128    # recurrent layer units
DROPOUT_RATE = 0.3    # fraction of units to drop for regularization
LEARNING_RATE = 0.001
EPOCHS        = 150
BATCH_SIZE    = 32

def build_model(arch='rnn'):
    """
    Build a text generation model.
    arch: 'rnn' | 'lstm' | 'gru'
    """
    recurrent_layers = {
        'rnn' : SimpleRNN(HIDDEN_UNITS, return_sequences=False),
        'lstm': LSTM(HIDDEN_UNITS, return_sequences=False),
        'gru' : GRU(HIDDEN_UNITS, return_sequences=False),
    }

    model = Sequential([
        # Layer 1 — Word Embeddings
        Embedding(
            input_dim=total_words,
            output_dim=EMBED_DIM,
            input_length=max_len - 1
        ),
        # Layer 2 — Recurrent layer (architecture-specific)
        recurrent_layers[arch],
        # Layer 3 — Regularization
        Dropout(DROPOUT_RATE),
        # Layer 4 — Output: probability over vocabulary
        Dense(total_words, activation='softmax')
    ], name=f"{arch.upper()}_TextGen")

    model.compile(
        loss='sparse_categorical_crossentropy',
        optimizer=Adam(learning_rate=LEARNING_RATE),
        metrics=['accuracy']
    )
    return model


rnn_model  = build_model('rnn')
lstm_model = build_model('lstm')
gru_model  = build_model('gru')

print("===== RNN Architecture =====")
rnn_model.summary()
print(f"\nParams comparison:")
print(f"  RNN  params : {rnn_model.count_params():,}")
print(f"  LSTM params : {lstm_model.count_params():,}")
print(f"  GRU  params : {gru_model.count_params():,}")

## 🏋️ Step 5: Training with Callbacks

**Theory — Callbacks for smarter training:**

| Callback | Purpose |
|----------|---------|
| `EarlyStopping` | Stop training when val_loss stops improving → prevents overfitting |
| `ReduceLROnPlateau` | Reduce learning rate when stuck → finer convergence |

**Why does LSTM/GRU converge better than RNN?**
- Vanishing gradient: in vanilla RNN, gradients become exponentially small
  as they backpropagate through many time steps.
- Gates in LSTM/GRU allow gradients to flow through unchanged via the
  cell state highway, enabling learning of longer patterns.

In [ ]:
def get_callbacks():
    return [
        EarlyStopping(
            monitor='loss', patience=15,
            restore_best_weights=True, verbose=0
        ),
        ReduceLROnPlateau(
            monitor='loss', factor=0.5,
            patience=8, min_lr=1e-5, verbose=0
        )
    ]

print("Training Vanilla RNN...")
rnn_history = rnn_model.fit(
    X, y,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    callbacks=get_callbacks(), verbose=0
)
print(f"  Done — {len(rnn_history.history['loss'])} epochs | "
      f"Final loss: {rnn_history.history['loss'][-1]:.4f} | "
      f"Acc: {rnn_history.history['accuracy'][-1]:.4f}")

print("Training LSTM...")
lstm_history = lstm_model.fit(
    X, y,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    callbacks=get_callbacks(), verbose=0
)
print(f"  Done — {len(lstm_history.history['loss'])} epochs | "
      f"Final loss: {lstm_history.history['loss'][-1]:.4f} | "
      f"Acc: {lstm_history.history['accuracy'][-1]:.4f}")

print("Training GRU...")
gru_history = gru_model.fit(
    X, y,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    callbacks=get_callbacks(), verbose=0
)
print(f"  Done — {len(gru_history.history['loss'])} epochs | "
      f"Final loss: {gru_history.history['loss'][-1]:.4f} | "
      f"Acc: {gru_history.history['accuracy'][-1]:.4f}")

## 📊 Step 6: Comprehensive Visualization

In [ ]:
fig = plt.figure(figsize=(18, 12))
fig.suptitle('RNN vs LSTM vs GRU — Training Analysis', fontsize=16, fontweight='bold')
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

COLORS = {'RNN': '#e74c3c', 'LSTM': '#2ecc71', 'GRU': '#3498db'}
histories = {
    'RNN': rnn_history,
    'LSTM': lstm_history,
    'GRU': gru_history
}

# ── Plot 1: Loss curves ────────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, :])
for name, hist in histories.items():
    ax1.plot(hist.history['loss'], label=name, color=COLORS[name], linewidth=2.5)
ax1.set(title='Training Loss Over Epochs', xlabel='Epoch', ylabel='Loss')
ax1.legend(fontsize=12)
ax1.grid(alpha=0.3)

# ── Plot 2: Accuracy curves ────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[1, 0])
for name, hist in histories.items():
    ax2.plot(hist.history['accuracy'], label=name, color=COLORS[name], linewidth=2)
ax2.set(title='Training Accuracy', xlabel='Epoch', ylabel='Accuracy')
ax2.legend()
ax2.grid(alpha=0.3)

# ── Plot 3: Final metrics bar chart ───────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 1])
names = list(COLORS.keys())
final_losses = [histories[n].history['loss'][-1] for n in names]
bars = ax3.bar(names, final_losses, color=list(COLORS.values()), width=0.5, edgecolor='black')
for bar, val in zip(bars, final_losses):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{val:.3f}', ha='center', va='bottom', fontweight='bold')
ax3.set(title='Final Training Loss', ylabel='Loss')
ax3.grid(axis='y', alpha=0.3)

# ── Plot 4: Parameter count comparison ────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 2])
param_counts = [
    rnn_model.count_params(),
    lstm_model.count_params(),
    gru_model.count_params()
]
bars2 = ax4.bar(names, param_counts, color=list(COLORS.values()), width=0.5, edgecolor='black')
for bar, val in zip(bars2, param_counts):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
             f'{val:,}', ha='center', va='bottom', fontweight='bold', fontsize=9)
ax4.set(title='Model Parameter Count', ylabel='# Parameters')
ax4.grid(axis='y', alpha=0.3)

plt.savefig('training_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved as training_analysis.png")

## ✍️ Step 7: Optimized Text Generation

**Theory — Two generation strategies:**

### Greedy (argmax)
```
Always pick the highest probability word.
→ Deterministic, repetitive, safe.
```

### Temperature Sampling
```
logits_scaled = logits / temperature
probabilities  = softmax(logits_scaled)
next_word      = random.choice(vocab, p=probabilities)

temperature > 1 → more random / creative
temperature < 1 → more confident / repetitive  
temperature = 1 → unmodified distribution
```
Temperature lets you control the **creativity vs coherence** tradeoff.

In [ ]:
def generate_text(model, seed_text, next_words=8, temperature=1.0):
    """
    Generate text by iteratively predicting the next word.

    Parameters
    ----------
    model       : trained Keras model
    seed_text   : starting phrase (string)
    next_words  : number of words to generate
    temperature : float > 0
                  1.0 = standard sampling
                  < 1 = conservative / repetitive
                  > 1 = creative / random
    """
    generated = seed_text

    for _ in range(next_words):
        # Tokenize current text
        token_list = tokenizer.texts_to_sequences([generated])[0]
        token_list = pad_sequences(
            [token_list], maxlen=max_len - 1, padding='pre'
        )

        # Get raw probabilities
        probs = model.predict(token_list, verbose=0)[0].astype('float64')

        if temperature == 0 or temperature is None:
            # Pure greedy
            predicted_idx = np.argmax(probs)
        else:
            # Temperature scaling
            probs = np.log(probs + 1e-10) / temperature
            probs = np.exp(probs)
            probs = probs / probs.sum()
            predicted_idx = np.random.choice(len(probs), p=probs)

        # Index → word
        next_word = idx_to_word.get(predicted_idx, '')
        if next_word and next_word != '<OOV>':
            generated += ' ' + next_word

    return generated


# ── Generate text samples ──────────────────────────────────────────────────────
seeds = ["deep learning", "recurrent neural", "lstm helps"]
temperatures = [0.5, 1.0, 1.5]

print("=" * 75)
print("TEXT GENERATION RESULTS")
print("=" * 75)

for seed in seeds:
    print(f"\n🌱 Seed: '{seed}'")
    print("-" * 60)
    for temp in temperatures:
        label = {0.5: 'Conservative', 1.0: 'Balanced', 1.5: 'Creative'}[temp]
        print(f"  Temperature {temp} ({label}):")
        for name, model in [('RNN', rnn_model), ('LSTM', lstm_model), ('GRU', gru_model)]:
            out = generate_text(model, seed, next_words=8, temperature=temp)
            print(f"    [{name}]  {out}")
        print()

## 🔁 Step 8: Top-k Sampling (Advanced)

**Theory — Top-k sampling:**
Instead of sampling from the entire vocabulary, restrict to the top-k
most probable words and renormalize. This prevents the model from
accidentally picking very rare/wrong words while still being creative.

In [ ]:
def generate_text_topk(model, seed_text, next_words=8, temperature=0.8, top_k=5):
    """
    Generate text using top-k sampling.
    Restricts prediction to the top_k most likely words,
    then samples with temperature from those k candidates.
    """
    generated = seed_text

    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([generated])[0]
        token_list = pad_sequences(
            [token_list], maxlen=max_len - 1, padding='pre'
        )

        probs = model.predict(token_list, verbose=0)[0].astype('float64')

        # Keep only top-k
        top_k_indices = np.argsort(probs)[-top_k:]
        top_k_probs   = probs[top_k_indices]

        # Apply temperature
        top_k_probs = np.log(top_k_probs + 1e-10) / temperature
        top_k_probs = np.exp(top_k_probs)
        top_k_probs /= top_k_probs.sum()

        predicted_idx = np.random.choice(top_k_indices, p=top_k_probs)
        next_word = idx_to_word.get(predicted_idx, '')
        if next_word and next_word != '<OOV>':
            generated += ' ' + next_word

    return generated


print("=" * 65)
print("TOP-K SAMPLING (k=5, temperature=0.8)")
print("=" * 65)

seed = "deep learning"
for name, model in [('RNN', rnn_model), ('LSTM', lstm_model), ('GRU', gru_model)]:
    out = generate_text_topk(model, seed, next_words=10, temperature=0.8, top_k=5)
    print(f"[{name}]  {out}")

## 📋 Step 9: Final Comparison Table

In [ ]:
print("\n" + "=" * 80)
print(" MODEL COMPARISON SUMMARY ".center(80, '='))
print("=" * 80)

metrics = {}
for name, model, hist in [
    ('Vanilla RNN', rnn_model, rnn_history),
    ('LSTM',        lstm_model, lstm_history),
    ('GRU',         gru_model,  gru_history)
]:
    metrics[name] = {
        'params'     : model.count_params(),
        'epochs_run' : len(hist.history['loss']),
        'final_loss' : hist.history['loss'][-1],
        'final_acc'  : hist.history['accuracy'][-1],
    }

header = f"{'Model':<15} {'Params':>10} {'Epochs':>8} {'Final Loss':>12} {'Final Acc':>12}"
print(header)
print("-" * len(header))
for name, m in metrics.items():
    print(f"{name:<15} {m['params']:>10,} {m['epochs_run']:>8} "
          f"{m['final_loss']:>12.4f} {m['final_acc']:>12.4f}")

print("\n" + "=" * 80)
print(" ARCHITECTURE QUICK REFERENCE ".center(80, '='))
print("=" * 80)
print("""
 Model       │ Gates         │ Memory Type          │ Best For
─────────────┼───────────────┼──────────────────────┼───────────────────────────
 Vanilla RNN │ None          │ Short-term only       │ Very short sequences
 LSTM        │ 3 (i, f, o)   │ Long + short term     │ Long paragraphs, dialogs
 GRU         │ 2 (r, z)      │ Long + short term     │ Fast training, similar NLP
""")

## 🎓 Step 10: Key Takeaways & Student Tasks

### ✅ What You Learned
1. **Tokenization** converts raw text into integer sequences the model can process.
2. **N-gram sequences** create supervised training pairs (context → next word).
3. **Embedding layers** learn dense semantic representations of words.
4. **Recurrent layers** process sequences by maintaining and updating a hidden state.
5. **Vanishing gradient** is why Vanilla RNN fails on long sequences — gradients
   become too small to update earlier weights.
6. **Gates** in LSTM/GRU solve this by controlling gradient flow selectively.
7. **Temperature** controls the creativity-coherence tradeoff during generation.
8. **Top-k sampling** prevents low-quality word selections while maintaining diversity.

### 🧪 Beginner Experiments
- [ ] Replace the corpus with Shakespeare, news articles, or song lyrics
- [ ] Change `EMBED_DIM` from 64 → 128 and observe accuracy improvement
- [ ] Change `HIDDEN_UNITS` from 128 → 256
- [ ] Generate 15 words instead of 8
- [ ] Try `temperature=0.3` vs `temperature=2.0` — what changes?

### 🔬 Intermediate Experiments
- [ ] Add a second LSTM/GRU layer (`return_sequences=True` on the first)
- [ ] Add `BatchNormalization` after the recurrent layer
- [ ] Use character-level tokenization instead of word-level
- [ ] Try `top_k=3` vs `top_k=10` in the top-k generator

### 🚀 Advanced Experiments
- [ ] Implement Nucleus (Top-p) sampling
- [ ] Replace embedding with pre-trained GloVe / FastText vectors
- [ ] Train on a large corpus (Project Gutenberg books)
- [ ] Compare with a simple Transformer / attention-based model